Example 1

In [1]:
import rapidsegment as rs
from prettytable import PrettyTable
import pandas as pd
from rapidsegment import UniversalDataLoader
from rapidsegment import StrategicSegmentBuilder
from rapidsegment import StrategicSegmentScore
import duckdb

In [2]:
print(f"RapidSegment version: {rs.__version__}")

RapidSegment version: 1.2.1


LOAD DATA

In [3]:
data = UniversalDataLoader(file_path=r"/workspaces/RapidSegment/Examples/train.csv", ).load()
print(f"Loaded as {type(data)} table for better performance ")
data.to_pandas().head()

2026-08-14 04:34:53,194 | INFO     | [data_loader.py:147] | 📂 Loading file: /workspaces/RapidSegment/Examples/train.csv (extension: .csv)


Loaded as <class 'pyarrow.lib.Table'> table for better performance 


,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0.0,15674932.0,Okwudilichukwu,668.0,France,Male,33.0,3.0,0.00,2.0,1.0,0.0,181449.97,0.0
1,1.0,15749177.0,Okwudiliolisa,627.0,France,Male,33.0,1.0,0.00,2.0,1.0,1.0,49503.50,0.0
2,2.0,15694510.0,Hsueh,678.0,France,Male,40.0,10.0,0.00,2.0,1.0,0.0,184866.69,0.0
3,3.0,15741417.0,Kao,581.0,France,Male,34.0,2.0,148882.54,1.0,1.0,1.0,84560.88,0.0
4,4.0,15766172.0,Chiemenam,716.0,Spain,Male,33.0,5.0,0.00,2.0,1.0,1.0,15068.83,0.0


Encoding Target Variable into Binary Format


1. Rule Mining using OptimalBinning as binning strategy (binning_method="optimal")
2. IV as variable selection metric (selection_metric='iv')
3. Repeated use of features are allowed (max_feature_reuse = 5)
4. Segment ranks on #event -> %response rate -> lift

Setup Experiment

In [4]:
param_grid = {'min_sample_size': [20000, 15000, 10000, 5000, 500],'min_lift': [3.0, 2.0, 1.5]}
builder = StrategicSegmentBuilder(target = 'Exited',
                                  min_sample_size = 100,
                                  min_lift = 1.0, 
                                  min_events  = 50, 
                                  top_n_vars = 10,
                                  max_segments = 10,
                                  param_grid = param_grid,
                                  enable_diversity = False, 
                                  max_feature_reuse = 5, 
                                  enable_1way = True,
                                  enable_2way = True,
                                  enable_3way = True,
                                  feature_groups = None,
                                  ignore_features = ['id','CustomerId','Surname'],
                                  sort_priority = 'rate_count_lift',
                                  binning_method="naive",
                                  selection_metric='iv',
                                  expand_log_mode='full',
                                  max_expansion_hops=1,)

Start Experiment

In [5]:
segments_df = builder.extract_segments(data)

2026-08-14 04:34:53,266 | INFO     | [builder.py:1095] | 🚀 Starting hierarchical segment extraction...
2026-08-14 04:34:53,267 | INFO     | [builder.py:1114] | 📂 Created temporary disk-backed DB at: experiments/segmentation_20260814_316fdc03.duckdb
2026-08-14 04:34:53,284 | INFO     | [builder.py:1129] | ⚙️ DuckDB Configured for Disk Spilling: Threads=4/4, MemoryLimit=12GB, TempDir=None
2026-08-14 04:34:53,285 | INFO     | [builder.py:1133] | 📊 Sort priority: rate_count_lift
2026-08-14 04:34:53,285 | INFO     | [builder.py:1134] | 📦 Binning method: naive (naive_bins=5)
2026-08-14 04:34:53,651 | INFO     | [builder.py:1189] | 📊 Dynamic Grid Search Enabled: 15 configurations.
2026-08-14 04:34:53,653 | INFO     | [builder.py:1197] | 🔒 Locking Original Base Rate: 21.16%
2026-08-14 04:34:53,654 | INFO     | [builder.py:1223] | 🔄 Iteration 1 | Remaining Volume: 165,034 | Base Rate: 21.16%
2026-08-14 04:34:53,655 | INFO     | [builder.py:279] | 🔍 Computing IV and bins for 10 features...
2026-

Build Experiment Results

In [6]:
final_eval = builder.evaluate_final_coverage(data)

2026-08-14 04:35:08,892 | INFO     | [builder.py:1632] | 📊 Evaluating final hierarchical coverage on original data...


Final Segment Report

In [7]:

table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------------------+--------------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate    |        lift        | cumulative_sample_capture | cumulative_event_capture |
+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------------------+--------------------------+
|   1.0   |    7391.0   |     5613.0    | 75.94371532945475  | 21.159882206090867 | 4.478471102924246  | 3.589042443137721  |     4.478471102924246     |    16.073422868760918    |
|   2.0   |    9502.0   |     6655.0    | 70.03788676068196  | 21.159882206090867 | 5.757601463940763  | 3.309937459884421  |     10.23607256686501     |    35.130723633343834    |
|   3.0   |    2379.0   |     1321.0    | 55.52753257671291  | 21.159882206090867 | 1.441521141

+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------------------+--------------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate    |        lift        | cumulative_sample_capture | cumulative_event_capture |
+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------------------+--------------------------+
|   1.0   |    3164.0   |     2773.0    | 87.64222503160556  | 21.159882206090867 | 1.9171807021583431 | 4.141905147580537  |     1.9171807021583431    |     7.94078061911171     |
|   2.0   |    5089.0   |     4149.0    | 81.52878758105719  | 21.159882206090867 | 3.0836070143122023 | 3.8529887258819024 |     5.000787716470546     |    19.82188368030698     |
|   3.0   |    1988.0   |     1439.0    | 72.38430583501005  | 21.159882206090867 | 1.2046002641879856 | 3.420827447431359  |     6.205387980658531     |    23.942613327224308    |
|   4.0   |    2542.0   |     1720.0    | 67.66325727773406  | 21.159882206090867 | 1.5402886677896677 | 3.197714269801427  |     7.745676648448199     |    28.868016379828756    |
|   5.0   |    1661.0   |     1055.0    | 63.515954244431065 | 21.159882206090867 | 1.0064592750584729 | 3.001715870901588  |     8.752135923506671     |    31.88912115918788     |
|   6.0   |    6895.0   |     3170.0    | 45.975344452501815 | 21.159882206090867 | 4.177926972623823  | 2.172759942835023  |     12.930062896130494    |    40.96675352939492     |
|   7.0   |    9305.0   |     4103.0    | 44.09457281031703  | 21.159882206090867 | 5.638232121865797  | 2.083876100105341  |     18.56829501799629     |    52.71613069499728     |
|   8.0   |    1155.0   |     375.0     | 32.467532467532465 | 21.159882206090867 | 0.6998557872923156 | 1.5343909834331069 |     19.268150805288606    |    53.78998310472209     |
|   0.0   |   133235.0  |    16137.0    | 12.111682365744736 | 21.159882206090867 | 80.73184919471139  | 0.5723889314590982 |           100.0           |          100.0           |
+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------------------+--------------------------+

Segment SQL definition

In [8]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   Age=[44.0, inf) & Geography=[Germany] & NumOfProducts=[-inf, 2.0)
SQL Filter: ("Age" >= 44.0) AND ("Geography" IN ('Germany')) AND ("NumOfProducts" < 2.0)
--------------------------------------------------
Segment ID: 2
Raw Rule:   Age=[43.0, inf) & Balance=[-inf, 83264.28) & NumOfProducts=[-inf, 2.0)
SQL Filter: ("Age" >= 43.0) AND ("Balance" < 83264.28) AND ("NumOfProducts" < 2.0)
--------------------------------------------------
Segment ID: 3
Raw Rule:   Age=[41.0, inf) & Geography=[Germany] & NumOfProducts=[-inf, 2.0)
SQL Filter: ("Age" >= 41.0) AND ("Geography" IN ('Germany')) AND ("NumOfProducts" < 2.0)
--------------------------------------------------
Segment ID: 4
Raw Rule:   Age=[41.0, inf) & Gender=[Female] & Geography=[Germany]
SQL Filter: ("Age" >= 41.0) AND ("Gender" IN ('Female')) AND ("Geography" IN ('Germany'))
--------------------------------------------------
Segment ID: 5
Raw Rule:   Age=[41.0, inf) & Gender=[Fe

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   Age=[44.0, inf) & Geography=[Germany] & NumOfProducts=[-inf, 2.0)
SQL Filter: (Age >= 44.0) AND (Geography IN ('Germany')) AND (NumOfProducts < 2.0)
--------------------------------------------------
Segment ID: 2
Raw Rule:   Age=[43.0, inf) & Balance=[-inf, 83264.28) & NumOfProducts=[-inf, 2.0)
SQL Filter: (Age >= 43.0) AND (Balance < 83264.28) AND (NumOfProducts < 2.0)
--------------------------------------------------
Segment ID: 3
Raw Rule:   Age=[41.0, inf) & Geography=[Germany] & NumOfProducts=[-inf, 2.0)
SQL Filter: (Age >= 41.0) AND (Geography IN ('Germany')) AND (NumOfProducts < 2.0)
--------------------------------------------------
Segment ID: 4
Raw Rule:   Age=[41.0, inf) & Gender=[Female] & Geography=[Germany]
SQL Filter: (Age >= 41.0) AND (Gender IN ('Female')) AND (Geography IN ('Germany'))
--------------------------------------------------
Segment ID: 5
Raw Rule:   Age=[41.0, inf) & Gender=[Female] & NumOfProducts=[-inf, 2.0)
SQL Filter: (Age >= 41.0) AND (Gender IN ('Female')) AND (NumOfProducts < 2.0)
--------------------------------------------------
Segment ID: 6
Raw Rule:   Gender=[Female] & Geography=[Germany] & NumOfProducts=[-inf, 2.0)
SQL Filter: (Gender IN ('Female')) AND (Geography IN ('Germany')) AND (NumOfProducts < 2.0)
--------------------------------------------------

Segment Meta informations

In [9]:

table = PrettyTable()
table.field_names = list(pd.DataFrame(segments_df).columns)
for _, row in pd.DataFrame(segments_df).iterrows():
    table.add_row(list(row))
print(table)

+------------+------------------------------------------------------------------------+---------------------------------------------------------------------------------------+-------+-------------------+--------------------+--------------------------+-----------------------+
| segment_id |                              rule_string                               |                                       sql_filter                                      | count |        rate       |        lift        | meta_applied_sample_size | meta_applied_min_lift |
+------------+------------------------------------------------------------------------+---------------------------------------------------------------------------------------+-------+-------------------+--------------------+--------------------------+-----------------------+
|     1      |   Age=[44.0, inf) & Geography=[Germany] & NumOfProducts=[-inf, 2.0)    |      ("Age" >= 44.0) AND ("Geography" IN ('Germany')) AND ("NumOfProducts" < 2.0)   

In [10]:
print(builder.explain_no_segments())

SEGMENT EXTRACTION DIAGNOSTIC REPORT
Segments Extracted : 6 / 10
Iterations Run     : 7
Stop Reason        : No candidate rules met the minimum grid thresholds (sample size / lift).
--------------------------------------------------------------------------------
Active Constraints:
  - min_sample_size : 100
  - min_lift        : 1.00x
  - min_events      : 50
  - selection_metric: iv

Feature Eligibility Summary (Iteration 7):
  - 6   feature(s): Eligible for Combination Search
  - 2   feature(s): Excluded (Max Feature Reuse Exceeded)
  - 2   feature(s): Excluded (IV is Zero/Invalid)

Candidate Funnel (Iteration 7):
  - 1-way candidates passing base criteria : 0
  - 2-way candidates passing base criteria : 0
  - 3-way candidates passing base criteria : 0
  - Total candidates before grid search   : 1,269
  - Candidates clearing grid filter       : 0



Selection Audit Trail for Variable

In [11]:
builder.explain_feature_journey("campaign")

📌 AUDIT TRAIL FOR FEATURE: 'campaign'
Iteration 1: Variable not present or was ignored.
Iteration 2: Variable not present or was ignored.
Iteration 3: Variable not present or was ignored.
Iteration 4: Variable not present or was ignored.
Iteration 5: Variable not present or was ignored.
Iteration 6: Variable not present or was ignored.
Iteration 7: Variable not present or was ignored.


Preparing the dataset for scoring and decile banding.
- Only score when atleast 10 segments are found

In [12]:
conn = duckdb.connect()
conn.register("predicted", mod_data)
predicted = conn.query("""
                        SELECT *, 
                        CASE WHEN (contact IN ('cellular')) AND (housing IN ('no'))
                        THEN 1 ELSE 0 END AS seg_1,
                        CASE WHEN (duration >= 551.50)
                        THEN 1 ELSE 0 END AS seg_2,
                        CASE WHEN (month IN ('feb', 'dec', 'sep', 'oct', 'mar'))
                        THEN 1 ELSE 0 END AS seg_3,
                        CASE WHEN (pdays >= 0.00 AND pdays < 213.00) AND (poutcome IN ('other', 'success'))
                        THEN 1 ELSE 0 END AS seg_4,                                                                                       
                        ROW_NUMBER() OVER () AS ID,
                        FROM predicted
""").df()
conn.close()

NameError: name 'mod_data' is not defined

In [ ]:
predicted.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,...,campaign,pdays,previous,poutcome,Target,seg_1,seg_2,seg_3,seg_4,ID
0,58.0,management,married,tertiary,no,2143.0,yes,no,unknown,5.0,...,1.0,-1.0,0.0,unknown,0,0,0,0,0,1
1,44.0,technician,single,secondary,no,29.0,yes,no,unknown,5.0,...,1.0,-1.0,0.0,unknown,0,0,0,0,0,2
2,33.0,entrepreneur,married,secondary,no,2.0,yes,yes,unknown,5.0,...,1.0,-1.0,0.0,unknown,0,0,0,0,0,3
3,47.0,blue-collar,married,unknown,no,1506.0,yes,no,unknown,5.0,...,1.0,-1.0,0.0,unknown,0,0,0,0,0,4
4,33.0,unknown,single,unknown,no,1.0,no,no,unknown,5.0,...,1.0,-1.0,0.0,unknown,0,0,0,0,0,5


Score the segments on the dataset and create decile bands


In [ ]:
scorer = StrategicSegmentScore(
    target_col="Target",
    primary_key="ID",
    segment_cols=["seg_1","seg_2",'seg_3','seg_4'],
)

Export Segment Score as JSON

In [ ]:
model_artifact = scorer.calculate_and_export_weights(predicted)

2026-08-12 14:00:31,440 | INFO     | [scorer.py:71] | 🚀 Initialising out‑of‑core DuckDB scorecard engine...
2026-08-12 14:00:31,702 | INFO     | [scorer.py:113] | 📊 Computing scorecard weights...
2026-08-12 14:00:31,703 | WARNING  | [scorer.py:157] | ⚠️ DECILE RESOLUTION WARNING: Only 4 distinct non-zero score values found across 4 segments. Splitting into 10 deciles will produce repeated thresholds (e.g., top 5 deciles may have identical scores). For smooth decile ranking, ensure the builder discovers at least 10 distinct segments (increase `max_segments`). Consider interpreting results as tiers rather than deciles.
2026-08-12 14:00:31,703 | INFO     | [scorer.py:171] | ⚡ Scoring population natively via SQL engine...
2026-08-12 14:00:31,728 | INFO     | [scorer.py:191] | 📉 Dataset Zero‑Inflation Rate: 88.30%
2026-08-12 14:00:31,729 | INFO     | [scorer.py:196] | 📈 Calibrating deciles across active populations...
2026-08-12 14:00:31,736 | INFO     | [scorer.py:244] | ✅ Scorecard export

View segment score and create final Decile based summary

In [ ]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 20
Segment: seg_2 | Weight: 46
Segment: seg_3 | Weight: 30
Segment: seg_4 | Weight: 48


In [ ]:
model_artifact.get("decile_min_thresholds")

{'1': 144,
 '2': 66,
 '3': 50,
 '4': 46,
 '5': 46,
 '6': 20,
 '7': 20,
 '8': 20,
 '9': 20,
 '10': 20}

In [ ]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 20 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 46 ELSE 0 END AS seg_2_weighted,
    CASE WHEN seg_3 = 1 THEN 30 ELSE 0 END AS seg_3_weighted,
    CASE WHEN seg_4 = 1 THEN 48 ELSE 0 END AS seg_4_weighted,
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted + seg_3_weighted + seg_4_weighted ) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=144 THEN 1
                    WHEN total_weight >= 66 THEN 2
                    WHEN total_weight >= 50 THEN 3
                    WHEN total_weight >= 46 THEN 4
                    WHEN total_weight >= 46 THEN 5
                    WHEN total_weight >= 20 THEN 6
                    WHEN total_weight >= 20 THEN 7
                    WHEN total_weight >= 20 THEN 8
                    WHEN total_weight >= 20 THEN 9
                    WHEN total_weight >= 20 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [ ]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(Target) AS events, 
                    (SUM(Target)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()
scored

,decile_band,count,events,response_rate
0,0,25008,618.0,2.471209
1,1,47,37.0,78.723404
2,2,3003,1600.0,53.280053
3,3,1863,507.0,27.214171
4,4,3340,1279.0,38.293413
5,6,11950,1248.0,10.443515
